In [1]:
import requests
from bs4 import BeautifulSoup
import urllib3
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import pandasql as ps

currentdate = datetime.now().strftime("%Y%m%d")

In [2]:
t1t2 = pd.read_csv('emission_factor_t1t2.csv')
t3 = pd.read_csv('inner_join_20250526.csv')

In [3]:
category = pd.concat([t1t2['กลุ่ม'],t3['Industrials']],axis=0,ignore_index=True)
name = pd.concat([t1t2['ชื่อ'],t3['Name']],axis=0,ignore_index=True)
unit = pd.concat([t1t2['หน่วย'],t3['Unit']],axis=0,ignore_index=True)
ef = pd.concat([t1t2['ค่าแฟคเตอร์ (kgCO2e)'],t3['EF']],axis=0,ignore_index=True)
reference = pd.concat([t1t2['ข้อมูลอ้างอิง'],t3['Company_name']],axis=0,ignore_index=True)
last_updated = pd.concat([t1t2['วันที่อัพเดท'],t3['ApproveDate']],axis=0,ignore_index=True)

# new table
table = pd.DataFrame({
    'Category': category,
    'Name': name,
    'Unit': unit,
    'Factor': ef,
    'Reference': reference,
    'Last_Updated': last_updated
})

In [ ]:
month_dict = {
    '01': 'Jan', '02': 'Feb', '03': 'Mar', '04': 'Apr',
    '05': 'May', '06': 'Jun', '07': 'Jul', '08': 'Aug',
    '09': 'Sep', '10': 'Oct', '11': 'Nov', '12': 'Dec'
}

def convert_month_to_name(value):
    if pd.isna(value):
        return value
    
    value = str(value)
    
    if ' ' in value and len(value.split(' ')) == 2:
        parts = value.split(' ')
        month_num = parts[0]
        year = parts[1]
        if month_num in month_dict:
            return f"{month_dict[month_num]} {year}"
    
    return value


def convert_unit(value):
    if pd.isna(value):
        return value
    
    value = str(value)
    if value == "1 กิโลกรัม":
        return "kg"
    return value


table['Unit'] = table['Unit'].apply(convert_unit)
table['Last_Updated'] = table['Last_Updated'].apply(convert_month_to_name)

0     Dec 2019
1     Dec 2019
2     Dec 2019
3     Dec 2019
4    July 2022
5     Dec 2019
6    July 2022
7     Dec 2019
8     Dec 2019
9     Dec 2019
Name: Last_Updated, dtype: object


In [6]:
table['Category'].unique()

array(['กลุ่มปิโตรเคมี', 'กลุ่มผลิตภัณฑ์จากก๊าซธรรมชาติ',
       'กลุ่มพลังงาน: เชื้อเพลิงเหลว และเชื้อเพลิงแข็ง', 'กลุ่มไฟฟ้า',
       'กลุ่มน้ำประปาและน้ำอุตสาหกรรม (Tap water)',
       'กลุ่มการขนส่งโดยรถบรรทุก (Truck Transportations) และขนส่งประเภทอื่น ๆ (Others)',
       'สิ่งทอ', 'กลุ่มอุตสาหกรรมยางธรรมชาติ (Natural rubber)',
       'กลุ่มอุตสาหกรรมโรงเลื่อยและโรงอบไม้ยางพารา (Wood Processing : Para-wood)',
       'ปาล์มน้ำมัน', 'กลุ่มอาหารสัตว์',
       'กลุ่มผลิตภัณฑ์ทางการเกษตรและอาหาร', 'กลุ่มปศุสัตว์',
       'กลุ่มผลิตภัณฑ์ที่ได้จากสัตว์และกลุ่มผลิตภัณฑ์ทางการเกษตร',
       'กลุ่มเครื่องจักรกลทางการเกษตร',
       'กลุ่มการจัดการมูลฝอยชุมชน และการปรับปรุงน้ำเสียชุมชน',
       'กลุ่มเยื่อและกระดาษ', 'กลุ่มเคมีภัณฑ์ (Chemicals)',
       'กลุ่มการฝังกลบขยะ', 'กลุ่มแก้วและกระจก',
       'กลุ่มไหมหัตถกรรม (Sericulture)', 'Stationary Combustion',
       'Mobile Combustion (On road)',
       'Mobile Combustion (Off road), Diesel',
       'Mobile Combustion (On road), Motor Gasoline

In [50]:
table.to_json(f'combine_{currentdate}.json',orient='records',indent=4,force_ascii=False)
with open(f'combine_{currentdate}.json', 'r', encoding='utf-8') as f:
    json_str = f.read()
json_str = json_str.replace('\\/', '/')
json_str = json_str.replace('null','"NULL"')

with open(f'combine_{currentdate}.json', 'w', encoding='utf-8') as f:
    f.write(json_str)